### Setup

In [1]:
from pathlib import Path
import sys
import torch
import torch.nn as nn

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        if (
            (candidate / "main.py").exists()
            and
            (candidate / "utils").exists()
            and
            (candidate / "thesis").exists()
        ):
            return candidate

    raise RuntimeError(
        "PyCIL project root not found."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


from thesis.backbones.resnet_attention import (
    resnet18_remote,
    resnet34_remote,
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "CUDA:",
    torch.cuda.is_available(),
)

Project root: C:\Skripsi\Project\PyCIL
PyTorch: 2.6.0+cu126
CUDA: True


### Ensure no CIFAR override


In [2]:
import inspect

from models.fetril import FeTrIL


source = inspect.getsource(
    FeTrIL.incremental_train
)

assert "RandomCrop(32" not in source
assert "_train_trsf" not in source

print(
    "FeTrIL transform override removed: PASSED"
)

FeTrIL transform override removed: PASSED


### Ensure backbone work on FeTriL

In [3]:
import torch

from thesis.backbones.variants import (
    get_all_backbone_variants,
)


device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)


base_args = {
    "dataset": "aid",

    "memory_size": 0,
    "fixed_memory": False,

    "device": [device],

    "batch_size": 2,
    "num_workers": 0,

    "seed": 1993,

    "fetril_classifier": "svm",

    "svm_c": 1.0,
    "svm_tol": 1e-4,

    "init_epochs": 1,
    "init_lr": 0.1,
    "init_weight_decay": 5e-4,

    "epochs": 1,
    "lr": 0.05,
    "weight_decay": 5e-4,
}


for name, backbone_args in (
    get_all_backbone_variants()
    .items()
):

    args = {
        **base_args,
        **backbone_args,
    }

    learner = FeTrIL(
        args
    )

    learner._network.update_fc(
        14
    )

    learner._network.eval()

    x = torch.randn(
        1,
        3,
        256,
        256,
    )

    with torch.no_grad():
        output = (
            learner._network(x)
        )

    assert (
        output["features"].shape
        == (1, 512)
    )

    assert (
        output["logits"].shape
        == (1, 14)
    )

    print(
        name,
        ": PASSED"
    )

r18_none : PASSED
r18_eca : PASSED
r18_cbam : PASSED
r18_procbam : PASSED
r34_none : PASSED
r34_eca : PASSED
r34_cbam : PASSED
r34_procbam : PASSED


### Validate freeze FeTriL

In [4]:
from thesis.backbones.variants import (
    get_backbone_variant,
)


for variant in [
    "r18_eca",
    "r18_cbam",
    "r18_procbam",
]:

    args = {
        **base_args,
        **get_backbone_variant(
            variant
        ),
    }

    learner = FeTrIL(args)

    learner._network.update_fc(
        14
    )

    learner._freeze_backbone()

    convnet_trainable = sum(
        p.requires_grad
        for p
        in learner
        ._network
        .convnet
        .parameters()
    )

    fc_trainable = sum(
        p.requires_grad
        for p
        in learner
        ._network
        .fc
        .parameters()
    )

    print(
        variant,
        "convnet:",
        convnet_trainable,
        "fc:",
        fc_trainable,
    )

    assert convnet_trainable == 0
    assert fc_trainable > 0


print(
    "FeTrIL frozen backbone: PASSED"
)

r18_eca convnet: 0 fc: 2
r18_cbam convnet: 0 fc: 2
r18_procbam convnet: 0 fc: 2
FeTrIL frozen backbone: PASSED


### Unit test SVM without training image

In [5]:
import numpy as np

from models.fetril import (
    FeatureDataset,
)


rng = np.random.RandomState(
    1993
)

num_classes = 14
feature_dim = 512


train_features = []
train_labels = []

test_features = []
test_labels = []


for c in range(
    num_classes
):

    center = np.zeros(
        feature_dim,
        dtype=np.float32,
    )

    center[c] = 10.0

    for _ in range(5):

        train_features.append(
            center
            + rng.normal(
                0,
                0.01,
                feature_dim,
            )
        )

        train_labels.append(c)

    for _ in range(2):

        test_features.append(
            center
            + rng.normal(
                0,
                0.01,
                feature_dim,
            )
        )

        test_labels.append(c)


train_set = FeatureDataset(
    np.asarray(
        train_features,
        dtype=np.float32,
    ),
    np.asarray(
        train_labels,
        dtype=np.int64,
    ),
)

test_set = FeatureDataset(
    np.asarray(
        test_features,
        dtype=np.float32,
    ),
    np.asarray(
        test_labels,
        dtype=np.int64,
    ),
)


args = {
    **base_args,
    **get_backbone_variant(
        "r18_none"
    ),
}

learner = FeTrIL(args)

learner._feature_trainset = (
    train_set
)

learner._feature_testset = (
    test_set
)

learner._train_svm(
    train_set,
    test_set,
)

pred, true = learner._eval_svm()

top1 = (
    pred[:, 0] == true
).mean()


print(
    "Synthetic SVM top1:",
    top1,
)

assert top1 > 0.99

print(
    "FeTrIL SVM path: PASSED"
)

Synthetic SVM top1: 1.0
FeTrIL SVM path: PASSED


### Validasi reproducibility seed

In [6]:
import random
import numpy as np
import torch

from trainer import _set_random


_set_random(1993)

a_python = random.random()
a_numpy = np.random.rand()
a_torch = torch.rand(1)


_set_random(1993)

b_python = random.random()
b_numpy = np.random.rand()
b_torch = torch.rand(1)


assert a_python == b_python
assert a_numpy == b_numpy
assert torch.equal(
    a_torch,
    b_torch,
)


print(
    "Training seed reproducibility: PASSED"
)

Training seed reproducibility: PASSED


## Full Fetril smoke test

### Configure smoke test

In [7]:
from pathlib import Path

import gc
import time

import numpy as np
import torch

from torch.utils.data import (
    WeightedRandomSampler,
)

from models.fetril import FeTrIL
from trainer import _set_random
from utils.data_manager import DataManager

from thesis.backbones.variants import (
    get_backbone_variant,
)

In [8]:
SMOKE_SEED = 1993

smoke_args = {
    # =========================
    # Dataset
    # =========================
    "dataset": "aid",

    "dataset_root": str(
        PROJECT_ROOT
        / "datasets/raw/AID"
    ),

    "split_root": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993"
    ),

    "task_config_path": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993/T5_tasks.json"
    ),

    "shuffle": False,

    # =========================
    # Method
    # =========================
    "model_name": "fetril",

    "memory_size": 0,
    "fixed_memory": False,

    "fetril_classifier": "svm",

    "svm_c": 1.0,
    "svm_tol": 1e-4,

    # =========================
    # Smoke training
    # =========================
    "init_epochs": 1,

    "init_lr": 0.1,

    "init_weight_decay": 5e-4,

    # Tidak dipakai canonical SVM
    # pada incremental tasks,
    # tetapi tetap disediakan.
    "epochs": 1,

    "lr": 0.05,

    "weight_decay": 5e-4,

    # AID 600x600.
    # Ini SMOKE batch size,
    # belum final experimental batch.
    "batch_size": 4,

    # Notebook Windows:
    # 0 paling aman.
    "num_workers": 0,

    # =========================
    # Reproducibility
    # =========================
    "seed": SMOKE_SEED,

    # BaseLearner membutuhkan
    # list torch.device.
    "device": [
        torch.device(
            "cuda:0"
            if torch.cuda.is_available()
            else "cpu"
        )
    ],

    # =========================
    # Backbone
    # =========================
    **get_backbone_variant(
        "r18_none"
    ),
}

### Create Data Manager

In [9]:
_set_random(
    SMOKE_SEED
)


data_manager = DataManager(
    dataset_name=(
        smoke_args["dataset"]
    ),

    shuffle=(
        smoke_args["shuffle"]
    ),

    seed=(
        SMOKE_SEED
    ),

    init_cls=14,
    increment=4,

    dataset_root=(
        smoke_args[
            "dataset_root"
        ]
    ),

    split_root=(
        smoke_args[
            "split_root"
        ]
    ),

    task_config_path=(
        smoke_args[
            "task_config_path"
        ]
    ),
)


print(
    "Tasks:",
    data_manager.nb_tasks
)

print(
    "Task sizes:",
    [
        data_manager
        .get_task_size(i)

        for i in range(
            data_manager.nb_tasks
        )
    ]
)


assert (
    data_manager.nb_tasks
    == 5
)

assert [
    data_manager
    .get_task_size(i)

    for i in range(
        data_manager.nb_tasks
    )
] == [
    14,
    4,
    4,
    4,
    4,
]


print(
    "Smoke DataManager: PASSED"
)

Tasks: 5
Task sizes: [14, 4, 4, 4, 4]
Smoke DataManager: PASSED


### Initialize Fetril

In [10]:
learner = FeTrIL(
    smoke_args
)


print(
    "Initial known classes:",
    learner._known_classes
)

print(
    "Current task:",
    learner._cur_task
)

print(
    "Feature dim:",
    learner.feature_dim
)


assert (
    learner._known_classes
    == 0
)

assert (
    learner._cur_task
    == -1
)

assert (
    learner.feature_dim
    == 512
)


print(
    "FeTrIL initialization: PASSED"
)

Initial known classes: 0
Current task: -1
Feature dim: 512
FeTrIL initialization: PASSED


### Reset VRAM Statistics ?

In [ ]:
if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


gc.collect()

### Run Task 0

In [ ]:
task0_start = time.perf_counter()
learner.incremental_train(data_manager)
task0_seconds = (time.perf_counter() - task0_start)

print(
    "Task 0 runtime:",
    round(
        task0_seconds / 60,
        2,
    ),
    "minutes",
)

### Validate Task 0

In [ ]:
print(
    "Current task:",
    learner._cur_task
)

print(
    "Total classes:",
    learner._total_classes
)

print(
    "Means:",
    len(
        learner._means
    )
)

print(
    "SVM classes:",
    learner
    ._svm_classifier
    .classes_
)

print(
    "Feature train shape:",
    learner
    ._feature_trainset
    .features
    .shape
)

print(
    "Feature test shape:",
    learner
    ._feature_testset
    .features
    .shape
)

print(
    "Train sampler:",
    type(
        learner
        .train_loader
        .sampler
    ).__name__,
)

print(
    "Exemplars:",
    learner.exemplar_size
)

In [ ]:
assert learner._cur_task == 0

assert (
    learner._total_classes
    == 14
)

assert (
    len(
        learner._means
    )
    == 14
)

assert np.array_equal(
    learner
    ._svm_classifier
    .classes_,

    np.arange(14),
)

assert (
    learner
    ._feature_trainset
    .features
    .shape[1]
    == 512
)

assert (
    learner
    ._feature_testset
    .features
    .shape[1]
    == 512
)

assert isinstance(
    learner
    .train_loader
    .sampler,

    WeightedRandomSampler,
)

assert (
    learner.exemplar_size
    == 0
)

print(
    "FeTrIL Task 0 structure: PASSED"
)

### Evaluate SVM Task 0

In [ ]:
task0_accy, _ = (
    learner.eval_task()
)


print(
    "Task 0 SVM grouped:"
)

print(
    task0_accy[
        "grouped"
    ]
)

print(
    "Task 0 top-1:",
    task0_accy[
        "top1"
    ]
)

print(
    "Task 0 top-5:",
    task0_accy[
        "top5"
    ]
)


assert (
    0.0
    <= task0_accy[
        "top1"
    ]
    <= 100.0
)

assert np.isfinite(
    task0_accy[
        "top1"
    ]
)


print(
    "Task 0 SVM evaluation: PASSED"
)

In [ ]:
learner.after_task()


assert (
    learner._known_classes
    == 14
)

print(
    "Task 0 after_task: PASSED"
)

### Check peak of VRAM initial task

In [ ]:
if torch.cuda.is_available():

    peak_allocated = (
        torch.cuda
        .max_memory_allocated()
        / 1024**3
    )

    peak_reserved = (
        torch.cuda
        .max_memory_reserved()
        / 1024**3
    )

    print(
        "Peak allocated:",
        round(
            peak_allocated,
            2,
        ),
        "GB",
    )

    print(
        "Peak reserved:",
        round(
            peak_reserved,
            2,
        ),
        "GB",
    )

### Run Task 1 seperately

In [ ]:
task1_start = time.perf_counter()


learner.incremental_train(
    data_manager
)


task1_seconds = (
    time.perf_counter()
    - task1_start
)


print(
    "Task 1 runtime:",
    round(
        task1_seconds / 60,
        2,
    ),
    "minutes",
)

### Validate task 1

In [ ]:
print(
    "Current task:",
    learner._cur_task
)

print(
    "Known before after_task:",
    learner._known_classes
)

print(
    "Total classes:",
    learner._total_classes
)

print(
    "Means:",
    len(
        learner._means
    )
)

print(
    "Relations:",
    learner._relations.shape
)

print(
    "Train loader:",
    learner.train_loader
)

print(
    "SVM classes:",
    learner
    ._svm_classifier
    .classes_
)

In [ ]:
assert learner._cur_task == 1

assert (
    learner._known_classes
    == 14
)

assert (
    learner._total_classes
    == 18
)

assert (
    len(
        learner._means
    )
    == 18
)

assert (
    learner._relations.shape
    == (14,)
)

assert (
    learner.train_loader
    is None
)

assert np.array_equal(
    learner
    ._svm_classifier
    .classes_,

    np.arange(18),
)

In [ ]:
trainable_backbone = sum(
    parameter.numel()

    for parameter
    in learner
    ._network
    .convnet
    .parameters()

    if parameter.requires_grad
)


print(
    "Trainable backbone params:",
    trainable_backbone
)


assert (
    trainable_backbone
    == 0
)

print(
    "FeTrIL Task 1 incremental "
    "behavior: PASSED"
)

### Check pseudo feature set

In [ ]:
feature_labels = (
    learner
    ._feature_trainset
    .labels
    .numpy()
)


unique_labels = np.unique(
    feature_labels
)


print(
    "Feature labels:",
    unique_labels
)

print(
    "Number of feature samples:",
    len(
        feature_labels
    )
)


assert np.array_equal(
    unique_labels,
    np.arange(18),
)


print(
    "Old + new feature classes: "
    "PASSED"
)

### Evaluate task 1

In [ ]:
task1_accy, _ = (
    learner.eval_task()
)


print(
    "Task 1 top-1:",
    task1_accy[
        "top1"
    ]
)

print(
    "Task 1 grouped:",
    task1_accy[
        "grouped"
    ]
)


assert np.isfinite(
    task1_accy[
        "top1"
    ]
)

learner.after_task()


assert (
    learner._known_classes
    == 18
)


print(
    "Task 1 evaluation: PASSED"
)

### Run the rest Task 2-4

In [ ]:
top1_curve = [
    task0_accy["top1"],
    task1_accy["top1"],
]

task_runtime_minutes = [
    task0_seconds / 60,
    task1_seconds / 60,
]

In [ ]:
for expected_task in range(
    2,
    data_manager.nb_tasks,
):

    start = time.perf_counter()

    learner.incremental_train(
        data_manager
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    accy, _ = (
        learner.eval_task()
    )

    expected_total = (
        data_manager
        .get_accumulate_tasksize(
            expected_task
        )
    )

    print()
    print(
        "=" * 60
    )

    print(
        "Task:",
        expected_task
    )

    print(
        "Total classes:",
        learner._total_classes
    )

    print(
        "Means:",
        len(
            learner._means
        )
    )

    print(
        "SVM classes:",
        len(
            learner
            ._svm_classifier
            .classes_
        )
    )

    print(
        "Top-1:",
        accy["top1"]
    )

    print(
        "Runtime:",
        round(
            elapsed / 60,
            2,
        ),
        "minutes",
    )


    assert (
        learner._cur_task
        == expected_task
    )

    assert (
        learner._total_classes
        == expected_total
    )

    assert (
        len(
            learner._means
        )
        == expected_total
    )

    assert (
        len(
            learner
            ._svm_classifier
            .classes_
        )
        == expected_total
    )

    assert (
        learner.train_loader
        is None
    )

    assert (
        learner.exemplar_size
        == 0
    )

    assert np.isfinite(
        accy["top1"]
    )


    top1_curve.append(
        accy["top1"]
    )

    task_runtime_minutes.append(
        elapsed / 60
    )


    learner.after_task()

### Final smoke validation

In [ ]:
print(
    "Top-1 curve:",
    top1_curve
)

print(
    "Task runtime minutes:",
    [
        round(x, 2)
        for x
        in task_runtime_minutes
    ]
)


average_incremental_accuracy = (
    float(
        np.mean(
            top1_curve
        )
    )
)

final_accuracy = (
    float(
        top1_curve[-1]
    )
)


print(
    "Smoke AIA:",
    round(
        average_incremental_accuracy,
        2,
    )
)

print(
    "Smoke final accuracy:",
    round(
        final_accuracy,
        2,
    )
)


assert (
    len(
        top1_curve
    )
    == 5
)

assert (
    learner._known_classes
    == 30
)

assert (
    learner._total_classes
    == 30
)

assert (
    len(
        learner._means
    )
    == 30
)

assert (
    len(
        learner._svm_accs
    )
    == 5
)

assert (
    learner.exemplar_size
    == 0
)


print(
    "FULL FeTrIL T5 smoke test: "
    "PASSED"
)

## Benchmark with 8 backbone and AID dataset on FeTriL

### Setup

In [11]:
import gc
import time

import numpy as np
import pandas as pd
import torch

from torch import optim
from torch.nn import functional as F
from torch.utils.data import (
    WeightedRandomSampler,
)

from models.fetril import FeTrIL

from thesis.backbones.variants import (
    get_all_backbone_variants,
)

from thesis.data.loaders import (
    build_train_loader,
)

from trainer import _set_random

### Take AID Task 0

In [ ]:
task0_classes = (
    data_manager.get_task_size(0)
)

task0_train_dataset = (
    data_manager.get_dataset(
        np.arange(
            0,
            task0_classes,
        ),
        source="train",
        mode="train",
    )
)


print(
    "Task 0 classes:",
    task0_classes
)

print(
    "Task 0 samples:",
    len(
        task0_train_dataset
    )
)


assert task0_classes == 14

assert (
    len(
        task0_train_dataset
    )
    == 3656
)


print(
    "Benchmark dataset: PASSED"
)

### Benchmark function



In [12]:
BENCH_BATCH_SIZE = 4
WARMUP_STEPS = 3
BENCH_STEPS = 30


def benchmark_fetril_variant(
    variant_name,
    backbone_config,
):

    _set_random(
        SMOKE_SEED
    )

    args = {
        **smoke_args,
        **backbone_config,

        "batch_size":
            BENCH_BATCH_SIZE,

        "num_workers":
            0,
    }


    learner = FeTrIL(
        args
    )

    network = (
        learner._network
    )

    network.update_fc(
        nb_classes=14
    )

    device = args[
        "device"
    ][0]

    network = network.to(
        device
    )

    network.train()


    loader = build_train_loader(
        dataset=(
            task0_train_dataset
        ),

        dataset_name="aid",

        batch_size=(
            BENCH_BATCH_SIZE
        ),

        num_workers=0,

        seed=(
            SMOKE_SEED
        ),
    )


    assert isinstance(
        loader.sampler,
        WeightedRandomSampler,
    )


    optimizer = optim.SGD(
        network.parameters(),

        lr=0.1,

        momentum=0.9,

        weight_decay=5e-4,
    )


    iterator = iter(loader)


    # =========================
    # Warm-up
    # =========================
    for _ in range(
        WARMUP_STEPS
    ):

        _, inputs, targets = next(
            iterator
        )

        targets = targets.long()

        inputs = inputs.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = network(
            inputs
        )["logits"]

        loss = F.cross_entropy(
            logits,
            targets,
        )

        loss.backward()

        optimizer.step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    # =========================
    # Timed section
    # =========================
    start = time.perf_counter()

    last_loss = None


    for _ in range(
        BENCH_STEPS
    ):

        _, inputs, targets = next(
            iterator
        )

        targets = targets.long()

        inputs = inputs.to(
            device,
            non_blocking=True,
        )

        targets = targets.to(
            device,
            non_blocking=True,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = network(
            inputs
        )["logits"]


        loss = F.cross_entropy(
            logits,
            targets,
        )


        loss.backward()

        optimizer.step()

        last_loss = float(
            loss.detach().cpu()
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    seconds_per_step = (
        elapsed
        / BENCH_STEPS
    )


    projected_epoch_seconds = (
        seconds_per_step
        * len(loader)
    )


    attention_grad_params = sum(
        1
        for name, parameter
        in network.named_parameters()

        if (
            ".attention." in name
            and
            parameter.grad is not None
        )
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    result = {
        "variant":
            variant_name,

        "batch_size":
            BENCH_BATCH_SIZE,

        "steps_per_epoch":
            len(loader),

        "sec_per_step":
            seconds_per_step,

        "projected_epoch_min":
            projected_epoch_seconds
            / 60,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "last_loss":
            last_loss,
    }


    assert np.isfinite(
        last_loss
    )


    attention_type = (
        backbone_config[
            "attention_type"
        ]
    )


    if attention_type == "none":

        assert (
            attention_grad_params
            == 0
        )

    else:

        assert (
            attention_grad_params
            > 0
        )


    del learner
    del network
    del optimizer
    del loader
    del iterator
    del inputs
    del targets
    del logits
    del loss


    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return result

### Run all 8 variants

In [ ]:
benchmark_results = []


variants = (
    get_all_backbone_variants()
)


for variant_name, config in (
    variants.items()
):

    print()
    print(
        "=" * 60
    )

    print(
        "Benchmark:",
        variant_name
    )


    result = (
        benchmark_fetril_variant(
            variant_name,
            config,
        )
    )


    benchmark_results.append(
        result
    )


    print(
        "sec/step:",
        round(
            result[
                "sec_per_step"
            ],
            4,
        )
    )

    print(
        "projected epoch:",
        round(
            result[
                "projected_epoch_min"
            ],
            2,
        ),
        "min",
    )

    print(
        "peak allocated:",
        round(
            result[
                "peak_allocated_gb"
            ],
            2,
        ),
        "GB",
    )

    print(
        "attention gradients:",
        result[
            "attention_grad_params"
        ],
    )

### Show runtime table

In [ ]:
benchmark_df = pd.DataFrame(
    benchmark_results
)


display(
    benchmark_df[
        [
            "variant",
            "sec_per_step",
            "projected_epoch_min",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "attention_grad_params",
        ]
    ]
)

### Calculate slowdown on R18 None

In [ ]:
baseline_step_time = float(
    benchmark_df.loc[
        benchmark_df[
            "variant"
        ]
        == "r18_none",

        "sec_per_step",
    ].iloc[0]
)


benchmark_df[
    "runtime_multiplier"
] = (
    benchmark_df[
        "sec_per_step"
    ]
    / baseline_step_time
)


display(
    benchmark_df[
        [
            "variant",
            "sec_per_step",
            "runtime_multiplier",
            "peak_allocated_gb",
        ]
    ]
)

### Estimate FeTriL runtime based on provided benchmark

In [ ]:
BASELINE_T5_MIN_HOURS = 5.0
BASELINE_T5_MAX_HOURS = 5.9

BASELINE_T10_MIN_HOURS = 4.3
BASELINE_T10_MAX_HOURS = 5.1

In [ ]:
benchmark_df[
    "T5_est_min_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T5_MIN_HOURS
)


benchmark_df[
    "T5_est_max_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T5_MAX_HOURS
)


benchmark_df[
    "T10_est_min_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T10_MIN_HOURS
)


benchmark_df[
    "T10_est_max_h"
] = (
    benchmark_df[
        "runtime_multiplier"
    ]
    * BASELINE_T10_MAX_HOURS
)


display(
    benchmark_df[
        [
            "variant",
            "runtime_multiplier",
            "T5_est_min_h",
            "T5_est_max_h",
            "T10_est_min_h",
            "T10_est_max_h",
            "peak_allocated_gb",
        ]
    ]
)

In [ ]:
total_min_hours = (
    benchmark_df[
        "T5_est_min_h"
    ].sum()
    +
    benchmark_df[
        "T10_est_min_h"
    ].sum()
)


total_max_hours = (
    benchmark_df[
        "T5_est_max_h"
    ].sum()
    +
    benchmark_df[
        "T10_est_max_h"
    ].sum()
)


print(
    "Estimated FeTrIL AID GPU-hours:"
)

print(
    round(
        total_min_hours,
        1,
    ),
    "-",
    round(
        total_max_hours,
        1,
    ),
    "hours",
)

## PASS

### Test dynamic rotation 256

In [13]:
from models.pa2s import PASS


pass_args = {
    **smoke_args,
    **get_backbone_variant(
        "r18_none"
    ),

    "model_name": "pass",

    "lambda_fkd": 10,
    "lambda_proto": 10,
    "temp": 0.1,

    "epochs": 1,

    "lr": 0.001,
    "weight_decay": 2e-4,

    "step_size": 45,
    "gamma": 0.1,

    "save_checkpoints": False,
}


pass_learner = PASS(
    pass_args
)


x = torch.randn(
    2,
    3,
    256,
    256,
)

y = torch.tensor(
    [3, 7],
    dtype=torch.long,
)


rx, ry = (
    pass_learner
    ._expand_rotations(
        x,
        y,
    )
)


print(
    "Input:",
    x.shape
)

print(
    "Rotated:",
    rx.shape
)

print(
    "Targets:",
    ry.tolist()
)


assert (
    rx.shape
    == (8, 3, 256, 256)
)

assert (
    ry.dtype
    == torch.long
)


print(
    "PASS 256 rotation: PASSED"
)

Input: torch.Size([2, 3, 256, 256])
Rotated: torch.Size([8, 3, 256, 256])
Targets: [12, 13, 14, 15, 28, 29, 30, 31]
PASS 256 rotation: PASSED


### Test AID 600x600

In [14]:
x = torch.randn(
    1,
    3,
    600,
    600,
)

y = torch.tensor(
    [5],
    dtype=torch.long,
)


rx, ry = (
    pass_learner
    ._expand_rotations(
        x,
        y,
    )
)


print(
    rx.shape
)

print(
    ry.tolist()
)


assert (
    rx.shape
    == (4, 3, 600, 600)
)

assert (
    ry.tolist()
    == [
        20,
        21,
        22,
        23,
    ]
)


print(
    "PASS 600 rotation: PASSED"
)

torch.Size([4, 3, 600, 600])
[20, 21, 22, 23]
PASS 600 rotation: PASSED


### Test classifier dimensionality

In [15]:
pass_learner = PASS(
    pass_args
)

pass_learner._network.update_fc(
    14 * 4
)


print(
    "PASS FC outputs:",
    pass_learner
    ._network
    .fc
    .out_features
)


assert (
    pass_learner
    ._network
    .fc
    .out_features
    == 56
)


print(
    "PASS rotation classifier: "
    "PASSED"
)

PASS FC outputs: 56
PASS rotation classifier: PASSED


### Test task 0 loss/backward

In [16]:
device = (
    pass_args["device"][0]
)


pass_learner._network = (
    pass_learner
    ._network
    .to(device)
)

pass_learner._network_module_ptr = (
    pass_learner._network
)

pass_learner._cur_task = 0
pass_learner._known_classes = 0
pass_learner._total_classes = 14


x = torch.randn(
    1,
    3,
    256,
    256,
    device=device,
)

y = torch.tensor(
    [2],
    dtype=torch.long,
    device=device,
)


x, y = (
    pass_learner
    ._expand_rotations(
        x,
        y,
    )
)


logits, clf, fkd, proto = (
    pass_learner
    ._compute_pass_loss(
        x,
        y,
    )
)


loss = (
    clf
    + fkd
    + proto
)


pass_learner._network.zero_grad()

loss.backward()


print(
    "Logits:",
    logits.shape
)

print(
    "Loss clf:",
    float(clf)
)

print(
    "Loss fkd:",
    float(fkd)
)

print(
    "Loss proto:",
    float(proto)
)


assert (
    logits.shape
    == (4, 56)
)

assert torch.isfinite(
    loss
)

assert (
    fkd.device
    == clf.device
)

assert (
    proto.device
    == clf.device
)


print(
    "PASS Task-0 backward: PASSED"
)

Logits: torch.Size([4, 56])
Loss clf: 8.726696014404297
Loss fkd: 0.0
Loss proto: 0.0
PASS Task-0 backward: PASSED


### Test loader policy

In [17]:
pass_train_dataset = (
    data_manager.get_dataset(
        np.arange(
            0,
            14,
        ),
        source="train",
        mode="train",
    )
)


loader = build_train_loader(
    dataset=(
        pass_train_dataset
    ),

    dataset_name="aid",

    batch_size=1,

    num_workers=0,

    seed=1993,
)


print(
    "PASS AID sampler:",
    type(
        loader.sampler
    ).__name__,
)


assert isinstance(
    loader.sampler,
    WeightedRandomSampler,
)


print(
    "PASS AID WRS: PASSED"
)

PASS AID sampler: WeightedRandomSampler
PASS AID WRS: PASSED


### Create new DataManager

In [18]:
from utils.data_manager import DataManager
from models.pa2s import PASS
from thesis.backbones.variants import (
    get_backbone_variant,
)
from trainer import _set_random

import gc
import time
import numpy as np
import torch

In [19]:
PASS_SMOKE_SEED = 1993


pass_smoke_args = {
    # =====================
    # Dataset
    # =====================
    "dataset": "aid",

    "dataset_root": str(
        PROJECT_ROOT
        / "datasets/raw/AID"
    ),

    "split_root": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993"
    ),

    "task_config_path": str(
        PROJECT_ROOT
        / "splits/aid/seed_1993/T5_tasks.json"
    ),

    "shuffle": False,

    # =====================
    # PASS
    # =====================
    "model_name": "pass",

    "memory_size": 0,
    "fixed_memory": False,

    "lambda_fkd": 10,
    "lambda_proto": 10,

    "temp": 0.1,

    # Smoke only
    "epochs": 1,

    "lr": 0.001,

    "weight_decay": 2e-4,

    "step_size": 45,

    "gamma": 0.1,

    # Raw batch.
    # Effective rotation batch = 4.
    "batch_size": 1,

    "num_workers": 0,

    "save_checkpoints": False,

    # =====================
    # Reproducibility
    # =====================
    "seed": PASS_SMOKE_SEED,

    "device": [
        torch.device(
            "cuda:0"
            if torch.cuda.is_available()
            else "cpu"
        )
    ],

    # =====================
    # Backbone
    # =====================
    **get_backbone_variant(
        "r18_none"
    ),
}

In [20]:
_set_random(
    PASS_SMOKE_SEED
)


pass_data_manager = DataManager(
    dataset_name="aid",

    shuffle=False,

    seed=PASS_SMOKE_SEED,

    init_cls=14,

    increment=4,

    dataset_root=(
        pass_smoke_args[
            "dataset_root"
        ]
    ),

    split_root=(
        pass_smoke_args[
            "split_root"
        ]
    ),

    task_config_path=(
        pass_smoke_args[
            "task_config_path"
        ]
    ),
)


print(
    "Tasks:",
    pass_data_manager.nb_tasks
)

print(
    "Task sizes:",
    [
        pass_data_manager
        .get_task_size(i)

        for i in range(
            pass_data_manager.nb_tasks
        )
    ]
)


assert (
    pass_data_manager.nb_tasks
    == 5
)

assert [
    pass_data_manager
    .get_task_size(i)

    for i in range(5)
] == [
    14,
    4,
    4,
    4,
    4,
]


print(
    "PASS DataManager: PASSED"
)

Tasks: 5
Task sizes: [14, 4, 4, 4, 4]
PASS DataManager: PASSED


### Initialize fresh PASS

In [21]:
pass_learner = PASS(
    pass_smoke_args
)


print(
    "Current task:",
    pass_learner._cur_task
)

print(
    "Known classes:",
    pass_learner._known_classes
)

print(
    "Feature dim:",
    pass_learner.feature_dim
)


assert (
    pass_learner._cur_task
    == -1
)

assert (
    pass_learner._known_classes
    == 0
)

assert (
    pass_learner.feature_dim
    == 512
)


print(
    "PASS initialization: PASSED"
)

Current task: -1
Known classes: 0
Feature dim: 512
PASS initialization: PASSED


In [22]:
print(
    "cur_task:",
    pass_learner._cur_task
)

print(
    "known_classes:",
    pass_learner._known_classes
)

print(
    "total_classes:",
    pass_learner._total_classes
)

print(
    "prototypes:",
    len(
        pass_learner._protos
    )
)


assert (
    pass_learner._cur_task
    == -1
)

assert (
    pass_learner._known_classes
    == 0
)

assert (
    pass_learner._total_classes
    == 0
)

assert (
    len(
        pass_learner._protos
    )
    == 0
)


print(
    "Fresh PASS state: PASSED"
)

cur_task: -1
known_classes: 0
total_classes: 0
prototypes: 0
Fresh PASS state: PASSED


### Task 0 Training

In [23]:
gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

In [24]:
print(
    "cur_task:",
    pass_learner._cur_task
)

print(
    "known_classes:",
    pass_learner._known_classes
)

print(
    "total_classes:",
    pass_learner._total_classes
)

cur_task: -1
known_classes: 0
total_classes: 0


In [25]:
import inspect

from models.pa2s import PASS


source = inspect.getsource(
    PASS.incremental_train
)

print(source)

print(
    "\n_cur_task += 1 occurrences:",
    source.count(
        "_cur_task += 1"
    )
)

print(
    "_cur_task = next_task occurrences:",
    source.count(
        "_cur_task = next_task"
    )
)

    def incremental_train(self, data_manager):
        self.data_manager = data_manager
    
        #Consistency guard
        next_task = (self._cur_task + 1)
        if next_task == 0:
            expected_known_classes = 0
        else:
            expected_known_classes = (data_manager.get_accumulate_tasksize(next_task - 1))
        if (self._known_classes != expected_known_classes):
            raise RuntimeError(
                "PASS task state is inconsistent. "
                f"About to start task {next_task}, "
                f"but _known_classes="
                f"{self._known_classes}, "
                f"expected "
                f"{expected_known_classes}. "
                "Did you forget to call "
                "after_task() after the "
                "previous task?"
            )

        self._cur_task = next_task
        self._total_classes = self._known_classes + \
            data_manager.get_task_size(self._cur_task)
        self._network.update_fc(self._

In [26]:
task0_start = (
    time.perf_counter()
)


pass_learner.incremental_train(
    pass_data_manager
)


task0_seconds = (
    time.perf_counter()
    - task0_start
)


print(
    "Task 0 runtime:",
    round(
        task0_seconds / 60,
        2,
    ),
    "minutes",
)

Task 0, Epoch 1/1 => Loss 4.247, Loss_clf 4.247, Loss_fkd 0.000, Loss_proto 0.000, Train_accy 4.16, Test_accy 7.77: 100%|██████████| 1/1 [04:31<00:00, 271.88s/it]


Task 0 runtime: 5.33 minutes


### Evaluate task 0

In [27]:
task0_cnn, task0_nme = (
    pass_learner.eval_task()
)


print(
    "Task 0 CNN top-1:",
    task0_cnn[
        "top1"
    ]
)

print(
    "Task 0 CNN grouped:",
    task0_cnn[
        "grouped"
    ]
)


if task0_nme is not None:

    print(
        "Task 0 NME top-1:",
        task0_nme[
            "top1"
        ]
    )


assert np.isfinite(
    task0_cnn[
        "top1"
    ]
)


print(
    "PASS Task 0 evaluation: PASSED"
)

Task 0 CNN top-1: 7.77
Task 0 CNN grouped: {'total': 7.77, '00-09': 3.53, '10-19': 16.9, 'old': 0, 'new': 7.77}
Task 0 NME top-1: 25.71
PASS Task 0 evaluation: PASSED


### Record peak Task 0 VRAM

In [28]:
if torch.cuda.is_available():

    task0_peak_allocated = (
        torch.cuda
        .max_memory_allocated()
        / 1024**3
    )

    task0_peak_reserved = (
        torch.cuda
        .max_memory_reserved()
        / 1024**3
    )


    print(
        "Task 0 peak allocated:",
        round(
            task0_peak_allocated,
            2,
        ),
        "GB",
    )

    print(
        "Task 0 peak reserved:",
        round(
            task0_peak_reserved,
            2,
        ),
        "GB",
    )

Task 0 peak allocated: 0.89 GB
Task 0 peak reserved: 1.26 GB


### after_task task 0

In [29]:
pass_learner.after_task()


print(
    "Known classes:",
    pass_learner._known_classes
)

print(
    "Old network exists:",
    pass_learner._old_network
    is not None
)


old_trainable = sum(
    p.numel()

    for p in (
        pass_learner
        ._old_network
        .parameters()
    )

    if p.requires_grad
)


print(
    "Old network trainable params:",
    old_trainable
)


print(
    "Old network pointer exists:",
    hasattr(
        pass_learner,
        "old_network_module_ptr",
    )
)


assert (
    pass_learner._known_classes
    == 14
)

assert (
    pass_learner._old_network
    is not None
)

assert (
    old_trainable
    == 0
)

assert hasattr(
    pass_learner,
    "old_network_module_ptr",
)


print(
    "PASS frozen teacher: PASSED"
)

Known classes: 14
Old network exists: True
Old network trainable params: 0
Old network pointer exists: True
PASS frozen teacher: PASSED


### Reset peak vram for task 1

In [30]:
gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

### Run task 1

In [31]:
task1_start = (
    time.perf_counter()
)


pass_learner.incremental_train(
    pass_data_manager
)


task1_seconds = (
    time.perf_counter()
    - task1_start
)


print(
    "Task 1 runtime:",
    round(
        task1_seconds / 60,
        2,
    ),
    "minutes",
)

Task 1, Epoch 1/1 => Loss 150.594, Loss_clf 4.776, Loss_fkd 106.473, Loss_proto 39.346, Train_accy 0.05, Test_accy 6.46: 100%|██████████| 1/1 [01:36<00:00, 96.41s/it]


Task 1 runtime: 1.81 minutes


### Validate incremental pass behavior

In [32]:
print(
    "Current task:",
    pass_learner._cur_task
)

print(
    "Known classes:",
    pass_learner._known_classes
)

print(
    "Total classes:",
    pass_learner._total_classes
)

print(
    "FC outputs:",
    pass_learner
    ._network
    .fc
    .out_features
)

print(
    "Prototypes:",
    len(
        pass_learner._protos
    )
)

print(
    "Last clf:",
    pass_learner
    ._last_loss_clf
)

print(
    "Last FKD:",
    pass_learner
    ._last_loss_fkd
)

print(
    "Last proto:",
    pass_learner
    ._last_loss_proto
)

Current task: 1
Known classes: 14
Total classes: 18
FC outputs: 72
Prototypes: 18
Last clf: 2.994779348373413
Last FKD: 328.25042724609375
Last proto: 39.01587677001953


In [33]:
assert (
    pass_learner._cur_task
    == 1
)

assert (
    pass_learner._known_classes
    == 14
)

assert (
    pass_learner._total_classes
    == 18
)

assert (
    pass_learner
    ._network
    .fc
    .out_features
    == 72
)

assert (
    len(
        pass_learner._protos
    )
    == 18
)

assert (
    pass_learner.exemplar_size
    == 0
)

assert np.isfinite(
    pass_learner
    ._last_loss_fkd
)

assert np.isfinite(
    pass_learner
    ._last_loss_proto
)

assert (
    pass_learner
    ._last_loss_fkd
    > 0
)

assert (
    pass_learner
    ._last_loss_proto
    > 0
)


print(
    "PASS Task 1 losses active: "
    "PASSED"
)

PASS Task 1 losses active: PASSED


### Current backbone must be trainable

In [34]:
current_trainable = sum(
    p.numel()

    for p in (
        pass_learner
        ._network
        .convnet
        .parameters()
    )

    if p.requires_grad
)


old_trainable = sum(
    p.numel()

    for p in (
        pass_learner
        ._old_network
        .convnet
        .parameters()
    )

    if p.requires_grad
)


print(
    "Current backbone trainable:",
    current_trainable
)

print(
    "Old backbone trainable:",
    old_trainable
)


assert (
    current_trainable
    > 0
)

assert (
    old_trainable
    == 0
)


print(
    "PASS student/teacher state: "
    "PASSED"
)

Current backbone trainable: 11176512
Old backbone trainable: 0
PASS student/teacher state: PASSED


### Task 1 VRAM

In [35]:
if torch.cuda.is_available():

    task1_peak_allocated = (
        torch.cuda
           .max_memory_allocated()
        / 1024**3
    )

    task1_peak_reserved = (
        torch.cuda
        .max_memory_reserved()
        / 1024**3
    )


    print(
        "Task 1 peak allocated:",
        round(
            task1_peak_allocated,
            2,
        ),
        "GB",
    )

    print(
        "Task 1 peak reserved:",
        round(
            task1_peak_reserved,
            2,
        ),
        "GB",
    )

Task 1 peak allocated: 1.09 GB
Task 1 peak reserved: 1.17 GB


### Evaluate task 1

In [36]:
task1_cnn, task1_nme = (
    pass_learner.eval_task()
)


print(
    "Task 1 CNN top-1:",
    task1_cnn[
        "top1"
    ]
)

print(
    "Task 1 grouped:",
    task1_cnn[
        "grouped"
    ]
)


assert np.isfinite(
    task1_cnn[
        "top1"
    ]
)


print(
    "PASS Task 1 evaluation: PASSED"
)

Task 1 CNN top-1: 6.46
Task 1 grouped: {'total': 6.46, '00-09': 10.42, '10-19': 1.72, 'old': 7.66, 'new': 1.72}
PASS Task 1 evaluation: PASSED


### After task task 1

In [37]:
pass_learner.after_task()

print(
    "After task:",
    pass_learner._cur_task
)

print(
    "Known classes:",
    pass_learner._known_classes
)

print(
    "Total classes:",
    pass_learner._total_classes
)


assert (
    pass_learner._known_classes
    == pass_learner._total_classes
)

After task: 1
Known classes: 18
Total classes: 18


### Run task 2-4

In [38]:
pass_top1_curve = [
    task0_cnn["top1"],
    task1_cnn["top1"],
]

pass_runtime_minutes = [
    task0_seconds / 60,
    task1_seconds / 60,
]

In [39]:
for expected_task in range(
    2,
    pass_data_manager.nb_tasks,
):

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    start = time.perf_counter()


    pass_learner.incremental_train(
        pass_data_manager
    )


    elapsed = (
        time.perf_counter()
        - start
    )


    cnn_accy, nme_accy = (
        pass_learner.eval_task()
    )


    expected_total = (
        pass_data_manager
        .get_accumulate_tasksize(
            expected_task
        )
    )


    print()
    print(
        "=" * 60
    )

    print(
        "Task:",
        expected_task
    )

    print(
        "Total classes:",
        pass_learner
        ._total_classes
    )

    print(
        "FC outputs:",
        pass_learner
        ._network
        .fc
        .out_features
    )

    print(
        "Prototypes:",
        len(
            pass_learner
            ._protos
        )
    )

    print(
        "FKD:",
        pass_learner
        ._last_loss_fkd
    )

    print(
        "Proto loss:",
        pass_learner
        ._last_loss_proto
    )

    print(
        "CNN top-1:",
        cnn_accy[
            "top1"
        ]
    )

    print(
        "Runtime:",
        round(
            elapsed / 60,
            2,
        ),
        "minutes",
    )


    assert (
        pass_learner._cur_task
        == expected_task
    )

    assert (
        pass_learner._total_classes
        == expected_total
    )

    assert (
        pass_learner
        ._network
        .fc
        .out_features
        == expected_total * 4
    )

    assert (
        len(
            pass_learner
            ._protos
        )
        == expected_total
    )

    assert (
        pass_learner.exemplar_size
        == 0
    )

    assert np.isfinite(
        pass_learner
        ._last_loss_fkd
    )

    assert np.isfinite(
        pass_learner
        ._last_loss_proto
    )


    pass_top1_curve.append(
        cnn_accy[
            "top1"
        ]
    )

    pass_runtime_minutes.append(
        elapsed / 60
    )


    pass_learner.after_task()

Task 2, Epoch 1/1 => Loss 46.559, Loss_clf 3.913, Loss_fkd 16.127, Loss_proto 26.519, Train_accy 0.00, Test_accy 3.76: 100%|██████████| 1/1 [01:59<00:00, 119.05s/it]



Task: 2
Total classes: 22
FC outputs: 88
Prototypes: 22
FKD: 6.98978328704834
Proto loss: 13.79977798461914
CNN top-1: 3.76
Runtime: 2.22 minutes


Task 3, Epoch 1/1 => Loss 134.311, Loss_clf 3.701, Loss_fkd 110.271, Loss_proto 20.339, Train_accy 2.68, Test_accy 6.93: 100%|██████████| 1/1 [02:01<00:00, 121.15s/it]



Task: 3
Total classes: 26
FC outputs: 104
Prototypes: 26
FKD: 202.59786987304688
Proto loss: 6.283251762390137
CNN top-1: 6.93
Runtime: 2.26 minutes


Task 4, Epoch 1/1 => Loss 43.908, Loss_clf 3.449, Loss_fkd 23.234, Loss_proto 17.225, Train_accy 3.06, Test_accy 2.75: 100%|██████████| 1/1 [02:07<00:00, 127.40s/it]



Task: 4
Total classes: 30
FC outputs: 120
Prototypes: 30
FKD: 11.375143051147461
Proto loss: 13.497537612915039
CNN top-1: 2.75
Runtime: 2.39 minutes


### Final PASS smoke validation

In [40]:
print(
    "PASS top-1 curve:",
    pass_top1_curve
)

print(
    "Runtime minutes:",
    [
        round(x, 2)

        for x in (
            pass_runtime_minutes
        )
    ]
)


smoke_aia = float(
    np.mean(
        pass_top1_curve
    )
)

smoke_final = float(
    pass_top1_curve[-1]
)


print(
    "Smoke AIA:",
    round(
        smoke_aia,
        2,
    )
)

print(
    "Smoke final:",
    round(
        smoke_final,
        2,
    )
)


assert (
    len(
        pass_top1_curve
    )
    == 5
)

assert (
    pass_learner._known_classes
    == 30
)

assert (
    pass_learner._total_classes
    == 30
)

assert (
    len(
        pass_learner._protos
    )
    == 30
)

assert (
    len(
        pass_learner._radiuses
    )
    == 30
)

assert (
    pass_learner.exemplar_size
    == 0
)


print(
    "FULL PASS AID T5 "
    "smoke test: PASSED"
)

PASS top-1 curve: [7.77, 6.46, 3.76, 6.93, 2.75]
Runtime minutes: [5.33, 1.81, 2.22, 2.26, 2.39]
Smoke AIA: 5.53
Smoke final: 2.75
FULL PASS AID T5 smoke test: PASSED


## Benchmark on 8 backbone and AID Dataset on PASS

### Dataset benchmark

In [41]:
pass_task0_dataset = (
    pass_data_manager.get_dataset(
        np.arange(0, 14),
        source="train",
        mode="train",
    )
)

pass_task1_dataset = (
    pass_data_manager.get_dataset(
        np.arange(14, 18),
        source="train",
        mode="train",
    )
)


print(
    "Task 0 samples:",
    len(pass_task0_dataset)
)

print(
    "Task 1 samples:",
    len(pass_task1_dataset)
)


assert (
    len(pass_task0_dataset)
    == 3656
)

print(
    "PASS benchmark datasets: PASSED"
)

Task 0 samples: 3656
Task 1 samples: 928
PASS benchmark datasets: PASSED


### Benchmark function

In [42]:
import gc
import time

import pandas as pd

from torch import optim
from torch.utils.data import (
    WeightedRandomSampler,
)

from thesis.backbones.variants import (
    get_all_backbone_variants,
)

from thesis.data.loaders import (
    build_train_loader,
)


PASS_BENCH_BATCH = 1

PASS_WARMUP_STEPS = 3

PASS_BENCH_STEPS = 30

In [43]:
def benchmark_pass_variant(
    variant_name,
    backbone_config,
    phase,
    batch_size=1,
    bench_steps=30,
):
    """
    phase:
        0 = initial task
        1 = incremental task
    """

    _set_random(
        PASS_SMOKE_SEED
    )

    args = {
        **pass_smoke_args,
        **backbone_config,

        "batch_size":
            batch_size,

        "num_workers":
            0,
    }

    learner = PASS(args)

    device = args[
        "device"
    ][0]

    # =====================================
    # Task 0 computational state
    # =====================================
    if phase == 0:

        learner._cur_task = 0
        learner._known_classes = 0
        learner._total_classes = 14

        learner._network.update_fc(
            14 * 4
        )

        dataset = (
            pass_task0_dataset
        )

    # =====================================
    # Task 1 computational state
    # =====================================
    elif phase == 1:

        # Build Task-0 classifier first.
        learner._network.update_fc(
            14 * 4
        )

        learner._network = (
            learner._network.to(
                device
            )
        )

        # Frozen teacher = state after Task 0.
        learner._old_network = (
            learner._network
            .copy()
            .freeze()
        )

        learner._old_network = (
            learner._old_network.to(
                device
            )
        )

        learner._old_network.eval()

        learner.old_network_module_ptr = (
            learner._old_network
        )

        # Expand student for Task 1.
        learner._network.update_fc(
            18 * 4
        )

        learner._cur_task = 1
        learner._known_classes = 14
        learner._total_classes = 18

        # Synthetic prototype state.
        # Hanya untuk benchmark compute path.
        rng = np.random.RandomState(
            PASS_SMOKE_SEED
        )

        learner._protos = [
            rng.normal(
                size=512
            ).astype(
                np.float32
            )

            for _ in range(14)
        ]

        learner._radius = 1.0

        dataset = (
            pass_task1_dataset
        )

    else:

        raise ValueError(
            "phase must be 0 or 1"
        )


    learner._network = (
        learner._network.to(
            device
        )
    )

    learner._network_module_ptr = (
        learner._network
    )

    learner._network.train()


    loader = build_train_loader(
        dataset=dataset,

        dataset_name="aid",

        batch_size=batch_size,

        num_workers=0,

        seed=(
            PASS_SMOKE_SEED
            + phase
        ),
    )


    assert isinstance(
        loader.sampler,
        WeightedRandomSampler,
    )


    optimizer = optim.Adam(
        learner
        ._network
        .parameters(),

        lr=0.001,

        weight_decay=2e-4,
    )


    iterator = iter(loader)


    # =====================================
    # Single optimization step
    # =====================================
    def run_step():

        _, inputs, targets = next(
            iterator
        )

        inputs = inputs.to(
            device,
            non_blocking=True,
        )

        targets = targets.to(
            device,
            dtype=torch.long,
            non_blocking=True,
        )

        inputs, targets = (
            learner
            ._expand_rotations(
                inputs,
                targets,
            )
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        (
            logits,
            loss_clf,
            loss_fkd,
            loss_proto,
        ) = learner._compute_pass_loss(
            inputs,
            targets,
        )

        loss = (
            loss_clf
            + loss_fkd
            + loss_proto
        )

        loss.backward()

        optimizer.step()

        return (
            float(
                loss.detach().cpu()
            )
        )


    # =====================================
    # Warm-up
    # =====================================
    for _ in range(
        PASS_WARMUP_STEPS
    ):
        run_step()


    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats()


    # =====================================
    # Timed benchmark
    # =====================================
    start = time.perf_counter()

    last_loss = None


    for _ in range(
        bench_steps
    ):

        last_loss = (
            run_step()
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    sec_per_step = (
        elapsed
        / bench_steps
    )


    projected_epoch_min = (
        sec_per_step
        * len(loader)
        / 60
    )


    if torch.cuda.is_available():

        peak_allocated = (
            torch.cuda
            .max_memory_allocated()
            / 1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            / 1024**3
        )

    else:

        peak_allocated = 0.0
        peak_reserved = 0.0


    attention_grad_params = sum(
        1

        for name, param
        in learner
        ._network
        .named_parameters()

        if (
            ".attention."
            in name

            and

            param.grad
            is not None
        )
    )


    result = {
        "variant":
            variant_name,

        "phase":
            (
                "task0"
                if phase == 0
                else "task1"
            ),

        "batch_size":
            batch_size,

        "samples":
            len(dataset),

        "steps_per_epoch":
            len(loader),

        "sec_per_step":
            sec_per_step,

        "projected_epoch_min":
            projected_epoch_min,

        "peak_allocated_gb":
            peak_allocated,

        "peak_reserved_gb":
            peak_reserved,

        "attention_grad_params":
            attention_grad_params,

        "last_loss":
            last_loss,
    }


    assert np.isfinite(
        last_loss
    )


    if (
        backbone_config[
            "attention_type"
        ]
        == "none"
    ):

        assert (
            attention_grad_params
            == 0
        )

    else:

        assert (
            attention_grad_params
            > 0
        )


    if phase == 1:

        teacher_trainable = sum(
            p.numel()

            for p in (
                learner
                ._old_network
                .parameters()
            )

            if p.requires_grad
        )

        assert (
            teacher_trainable
            == 0
        )


    del learner
    del optimizer
    del loader
    del iterator

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return result

### Run 8 variants x 2 phases

In [44]:
pass_benchmark_results = []


variants = (
    get_all_backbone_variants()
)


for variant_name, config in (
    variants.items()
):

    for phase in [
        0,
        1,
    ]:

        print()
        print(
            "=" * 60
        )

        print(
            variant_name,
            "Task",
            phase,
        )


        result = (
            benchmark_pass_variant(
                variant_name,
                config,
                phase=phase,
                batch_size=(
                    PASS_BENCH_BATCH
                ),
                bench_steps=(
                    PASS_BENCH_STEPS
                ),
            )
        )


        pass_benchmark_results.append(
            result
        )


        print(
            "sec/step:",
            round(
                result[
                    "sec_per_step"
                ],
                4,
            )
        )

        print(
            "projected epoch:",
            round(
                result[
                    "projected_epoch_min"
                ],
                2,
            ),
            "min",
        )

        print(
            "peak allocated:",
            round(
                result[
                    "peak_allocated_gb"
                ],
                2,
            ),
            "GB",
        )


r18_none Task 0
sec/step: 0.0685
projected epoch: 4.17 min
peak allocated: 1.0 GB

r18_none Task 1
sec/step: 0.0826
projected epoch: 1.28 min
peak allocated: 1.18 GB

r18_eca Task 0
sec/step: 0.0706
projected epoch: 4.3 min
peak allocated: 1.07 GB

r18_eca Task 1
sec/step: 0.0871
projected epoch: 1.35 min
peak allocated: 1.26 GB

r18_cbam Task 0
sec/step: 0.0883
projected epoch: 5.38 min
peak allocated: 1.16 GB

r18_cbam Task 1
sec/step: 0.1136
projected epoch: 1.76 min
peak allocated: 1.35 GB

r18_procbam Task 0
sec/step: 0.0939
projected epoch: 5.72 min
peak allocated: 1.16 GB

r18_procbam Task 1
sec/step: 0.121
projected epoch: 1.87 min
peak allocated: 1.35 GB

r34_none Task 0
sec/step: 0.1094
projected epoch: 6.67 min
peak allocated: 1.39 GB

r34_none Task 1
sec/step: 0.1386
projected epoch: 2.14 min
peak allocated: 1.6 GB

r34_eca Task 0
sec/step: 0.1175
projected epoch: 7.16 min
peak allocated: 1.53 GB

r34_eca Task 1
sec/step: 0.1492
projected epoch: 2.31 min
peak allocated: 1.

### Result table

In [45]:
pass_benchmark_df = (
    pd.DataFrame(
        pass_benchmark_results
    )
)


display(
    pass_benchmark_df[
        [
            "variant",
            "phase",
            "sec_per_step",
            "projected_epoch_min",
            "peak_allocated_gb",
            "peak_reserved_gb",
            "attention_grad_params",
        ]
    ]
)

,variant,phase,sec_per_step,projected_epoch_min,peak_allocated_gb,peak_reserved_gb,attention_grad_params
0,r18_none,task0,0.068473,4.172274,0.995542,1.294922,0
1,r18_none,task1,0.082580,1.277233,1.178578,1.343750,0
2,r18_eca,task0,0.070611,4.302561,1.073081,1.468750,8
3,r18_eca,task1,0.087073,1.346726,1.263161,1.429688,8
4,r18_cbam,task0,0.088276,5.378929,1.156449,1.544922,56
5,r18_cbam,task1,0.113604,1.757079,1.351460,1.505859,56
6,r18_procbam,task0,0.093923,5.723011,1.158780,1.548828,88
7,r18_procbam,task1,0.120954,1.870756,1.350524,1.509766,88
8,r34_none,task0,0.109424,6.667572,1.388736,1.753906,0
9,r34_none,task1,0.138639,2.144286,1.602658,1.812500,0


### Find final batch size

In [46]:
capacity_results = []


heavy_config = (
    get_backbone_variant(
        "r34_procbam"
    )
)


for batch in [
    2,
    4,
]:

    print()
    print(
        "=" * 60
    )

    print(
        "Testing raw batch:",
        batch
    )


    try:

        result = (
            benchmark_pass_variant(
                "r34_procbam",
                heavy_config,
                phase=1,
                batch_size=batch,
                bench_steps=10,
            )
        )

        capacity_results.append(
            result
        )


        print(
            "PASSED"
        )

        print(
            "sec/step:",
            round(
                result[
                    "sec_per_step"
                ],
                4,
            )
        )

        print(
            "peak allocated:",
            round(
                result[
                    "peak_allocated_gb"
                ],
                2,
            ),
            "GB",
        )

        print(
            "peak reserved:",
            round(
                result[
                    "peak_reserved_gb"
                ],
                2,
            ),
            "GB",
        )


    except RuntimeError as e:

        if (
            "out of memory"
            in str(e).lower()
        ):

            print(
                "OOM at batch",
                batch,
            )

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        else:
            raise


Testing raw batch: 2
PASSED
sec/step: 0.3846
peak allocated: 3.26 GB
peak reserved: 3.46 GB

Testing raw batch: 4
PASSED
sec/step: 8.3451
peak allocated: 5.95 GB
peak reserved: 6.43 GB


## SSRE